In [1]:
import pandas as pd
import numpy as np
import os
import glob
from IPython.display import display

# Configurações
pd.set_option('display.max_columns', None)
print("Bibliotecas importadas.")

Bibliotecas importadas.


In [2]:
# O caminho base é o diretório atual
base_path = '.' 

# Lista de pastas de ataque
attack_folders = [
    'analysis',
    'dos',
    'exploits',
    'fuzzers',
    'reconnaissance'
]

# Lista de todos os diretórios a serem processados
data_paths = {
    'Normal': os.path.join(base_path, 'normal')
}

# Adiciona os caminhos de ataque ao dicionário
for attack in attack_folders:
    data_paths[attack.capitalize()] = os.path.join(base_path, 'ataque', attack)

print("Caminhos de dados definidos:")
print(data_paths)

# Lista para armazenar todos os DataFrames
all_dataframes = []

# Loop para encontrar e ler todos os arquivos '*_features_processed.csv'
print("\nIniciando leitura dos arquivos '_features_processed.csv'...")

for category_name, folder_path in data_paths.items():
    if not os.path.isdir(folder_path):
        print(f"Aviso: Diretório não encontrado, pulando: {folder_path}")
        continue
    
    # Usa glob para encontrar o arquivo '*_features_processed.csv' dentro da pasta
    search_pattern = os.path.join(folder_path, '*_features_processed.csv')
    found_files = glob.glob(search_pattern)
    
    if not found_files:
        print(f"Aviso: Nenhum arquivo '_features_processed.csv' encontrado em: {folder_path}")
        continue
    
    # Assume que há apenas um arquivo _features_processed.csv por pasta
    file_path = found_files[0]
    
    try:
        print(f"Lendo: {file_path} ...")
        df_temp = pd.read_csv(file_path, low_memory=False)
        all_dataframes.append(df_temp)
    except Exception as e:
        print(f"Erro ao ler {file_path}: {e}")

print("\nLeitura de todos os arquivos concluída.")

Caminhos de dados definidos:
{'Normal': '.\\normal', 'Analysis': '.\\ataque\\analysis', 'Dos': '.\\ataque\\dos', 'Exploits': '.\\ataque\\exploits', 'Fuzzers': '.\\ataque\\fuzzers', 'Reconnaissance': '.\\ataque\\reconnaissance'}

Iniciando leitura dos arquivos '_features_processed.csv'...
Lendo: .\normal\normal_features_processed.csv ...
Lendo: .\ataque\analysis\analysis_features_processed.csv ...
Lendo: .\ataque\dos\dos_features_processed.csv ...
Lendo: .\ataque\exploits\exploits_features_processed.csv ...
Lendo: .\ataque\fuzzers\fuzzers_features_processed.csv ...
Lendo: .\ataque\reconnaissance\reconnaissance_features_processed.csv ...

Leitura de todos os arquivos concluída.


In [4]:
if not all_dataframes:
    print("ERRO: Nenhum DataFrame foi carregado. Não é possível continuar.")
else:
    # Juntar todos os DataFrames em um único dataset
    print("Concatenando todos os DataFrames...")
    final_dataset = pd.concat(all_dataframes, ignore_index=True)

    print("\nIniciando limpeza de valores NaN...")
    print(f"Total de valores nulos (NaN) antes da limpeza: {final_dataset.isnull().sum().sum()}")

    # Definir colunas com base no NUSW-NB15_features.csv
    # Tipos nominais (preencher com '-')
    # Inclui: nominal, Binary, binary
    nominal_cols = [
        'srcip', 'dstip', 'proto', 'state', 'service', 
        'is_sm_ips_ports', 'is_ftp_login', 
        'attack_cat', 'Label'
    ]
    
    # Tipos numéricos (preencher com 0), conforme arquivo-guia
    # Inclui: integer, Float, Timestamp
    numeric_cols = [
        'sport', 'dsport', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 
        'sloss', 'dloss', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 
        'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 
        'response_body_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 
        'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'ct_state_ttl', 
        'ct_flw_http_mthd', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 
        'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 
        'ct_dst_sport_ltm', 'ct_dst_src_ltm'
    ]

    # Identificar quais dessas colunas realmente existem no nosso DataFrame
    # (Pode ser que alguma coluna não tenha sido capturada ou gerada)
    nominal_cols_to_fill = [col for col in nominal_cols if col in final_dataset.columns]
    numeric_cols_to_fill = [col for col in numeric_cols if col in final_dataset.columns]
    
    # Substituir valores infinitos (inf) por 0 (APENAS em colunas numéricas)
    # Fazemos isso primeiro para garantir que 'inf' não seja tratado como string.
    if numeric_cols_to_fill:
         final_dataset[numeric_cols_to_fill] = final_dataset[numeric_cols_to_fill].replace([np.inf, -np.inf], 0)
    print("Valores infinitos (inf) em colunas numéricas substituídos por 0.")

    # Padronizar todos os valores "null-like" (incluindo strings vazias) para np.nan
    # Isso força strings vazias (''), 'None', 'NULL', etc., a se tornarem np.nan
    # para que o fillna() possa pegá-los.
    null_like_strings = ['', ' ', 'None', 'NULL', 'N/A', 'nan', 'NaN', 'undefined', '-']
    final_dataset.replace(null_like_strings, np.nan, inplace=True)
    print(f"Strings 'null-like' (incluindo '') convertidas para np.nan.")
    print(f"Total de NaNs para preencher agora: {final_dataset.isnull().sum().sum()}")
    
    # Aplicar preenchimento de NaN (agora que TODOS os nulos são np.nan)
    if nominal_cols_to_fill:
        final_dataset[nominal_cols_to_fill] = final_dataset[nominal_cols_to_fill].fillna('-')
        print(f"Valores NaN em {len(nominal_cols_to_fill)} colunas nominais/binárias preenchidos com '-'.")

    if numeric_cols_to_fill:
        final_dataset[numeric_cols_to_fill] = final_dataset[numeric_cols_to_fill].fillna(0)
        print(f"Valores NaN em {len(numeric_cols_to_fill)} colunas numéricas preenchidos com 0.")

    # Verificação final de NaNs restantes
    remaining_nans = final_dataset.isnull().sum().sum()
    print(f"Total de valores nulos (NaN) após a limpeza final: {remaining_nans}")
    
    if remaining_nans > 0:
        print("\nATENÇÃO: Ainda existem NaNs no DataFrame! Colunas com NaNs:")
        print(final_dataset.isnull().sum()[final_dataset.isnull().sum() > 0])

    
    # Balanceamento de amostras: Reduzir classe majoritária "Reconnaissance"

    print("\nBalanceando a classe 'Reconnaissance' para reduzir desbalanceamento...")

    # Contar instâncias por classe
    class_counts = final_dataset['attack_cat'].value_counts()
    print("Contagem original de classes:")
    print(class_counts)

    # Escolher o número alvo para Reconnaissance (ex.: 10% do total original)
    target_fraction = 0.010  # Mantenha apenas 10% da classe Reconnaissance
    recon_total = class_counts.get('Reconnaissance', 0)
    target_size = int(recon_total * target_fraction)

    if recon_total > 0:
        # Filtrar Reconnaissance
        df_recon = final_dataset[final_dataset['attack_cat'] == 'Reconnaissance']
        df_other = final_dataset[final_dataset['attack_cat'] != 'Reconnaissance']

        print(f"Reduzindo Reconnaissance de {recon_total} → {target_size} amostras...")

        # Fazer amostragem estratificada aleatória (random_state para reprodutibilidade)
        df_recon_sampled = df_recon.sample(n=target_size, random_state=42)

        # Reunir o dataset balanceado
        final_dataset = pd.concat([df_other, df_recon_sampled], ignore_index=True)

        print("Distribuição após balanceamento:")
        print(final_dataset['attack_cat'].value_counts())
    else:
        print("Nenhuma amostra de Reconnaissance encontrada — nada a balancear.")

    
    # Embaralhar o dataset final
    # Isso quebra qualquer ordem cronológica da captura
    print("\nEmbaralhando o dataset final...")
    final_dataset = final_dataset.sample(frac=1, random_state=42).reset_index(drop=True)
    print("##################################")
    print(final_dataset.isnull().values.any())

    # Salvar o dataset consolidado
    output_filename = 'ubuntu_server_dataset_final.csv'
    print("#############################################################################")
    print(final_dataset.isnull().values.any())
    print("#############################################################################")
    final_dataset.to_csv(output_filename, index=False)
    print(f"\nDataset consolidado e limpo salvo com sucesso como: {output_filename}")
    # 4. Verificação Final
    print("\n--- Informações do Dataset Final Consolidado ---")
    final_dataset.info()
    
    print("\n--- Amostra do Dataset Final (5 primeiras linhas) ---")
    display(final_dataset.head())
    
    print("\n--- Distribuição de Classes no Dataset Final ---")
    # Mostra a contagem e a porcentagem de cada categoria
    print(final_dataset['attack_cat'].value_counts())
    print("\nDistribuição Percentual:")
    print(final_dataset['attack_cat'].value_counts(normalize=True) * 100)

Concatenando todos os DataFrames...

Iniciando limpeza de valores NaN...
Total de valores nulos (NaN) antes da limpeza: 2024638
Valores infinitos (inf) em colunas numéricas substituídos por 0.
Strings 'null-like' (incluindo '') convertidas para np.nan.
Total de NaNs para preencher agora: 2024638
Valores NaN em 6 colunas nominais/binárias preenchidos com '-'.
Valores NaN em 25 colunas numéricas preenchidos com 0.
Total de valores nulos (NaN) após a limpeza final: 0

Balanceando a classe 'Reconnaissance' para reduzir desbalanceamento...
Contagem original de classes:
attack_cat
Reconnaissance    1966157
DoS                 52304
Exploits            35189
Fuzzers              7670
Normal               1480
Analysis              353
Name: count, dtype: int64
Reduzindo Reconnaissance de 1966157 → 19661 amostras...
Distribuição após balanceamento:
attack_cat
DoS               52304
Exploits          35189
Reconnaissance    19661
Fuzzers            7670
Normal             1480
Analysis        

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.000082,tcp,-,REJ,1,1,0.0,0.0,24390.243902,0,0,0.000000,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,0,0,0.0,0.0,1,100,100,1,1,100,0,0,0,100,1,0,Reconnaissance,1
1,0.015178,tcp,http,SH,5,0,452.0,0.0,329.424167,0,0,238239.557254,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,90,0,1.0,0.0,100,100,100,100,1,100,0,0,100,100,1,0,Exploits,1
2,0.004186,tcp,http,SH,5,0,442.0,0.0,1194.457716,0,0,844720.496894,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,88,0,1.0,0.0,100,100,100,100,1,100,0,0,100,100,1,0,Exploits,1
3,0.000295,tcp,-,S0,2,0,0.0,0.0,6779.661017,0,0,0.000000,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,0,0,0.0,0.0,100,100,100,100,1,100,0,0,0,100,1,0,DoS,1
4,0.000098,tcp,-,REJ,1,1,0.0,0.0,20408.163265,0,0,0.000000,0.0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,0,0,0.0,0.0,1,100,100,1,1,100,0,0,0,100,1,0,Reconnaissance,1



--- Distribuição de Classes no Dataset Final ---
attack_cat
DoS               52304
Exploits          35189
Reconnaissance    19661
Fuzzers            7670
Normal             1480
Analysis            353
Name: count, dtype: int64

Distribuição Percentual:
attack_cat
DoS               44.835715
Exploits          30.164499
Reconnaissance    16.853682
Fuzzers            6.574830
Normal             1.268677
Analysis           0.302597
Name: proportion, dtype: float64
